# Лекция: Разведочный анализ данных (EDA)

В этом ноутбуке теория сопровождается **небольшими, но конкретными примерами** работы с данными.  
Главная цель — не просто показать инструменты EDA, а объяснить:

- **зачем** выполняется каждый шаг;
- **какие решения** он помогает принять;
- **как** результаты EDA влияют на последующее построение модели машинного обучения.

---

## План лекции

1. Что такое EDA и зачем он нужен  
2. Первичное знакомство с данными  
3. Пропуски и некорректные значения  
4. Одномерный анализ признаков  
5. Выбросы  
6. Связи между признаками  
7. Связь признаков с целевой переменной  
8. Дисбаланс классов  
9. Как EDA помогает при построении модели  
10. Итоги

## 1. Что такое EDA и зачем он нужен

**Разведочный анализ данных (Exploratory Data Analysis, EDA)** — это этап изучения набора данных **до обучения модели**.

### Почему EDA важен

Если сразу перейти к обучению модели, можно столкнуться с проблемами:

- в данных есть пропуски;
- часть значений записана некорректно;
- признаки сильно коррелируют друг с другом;
- есть выбросы, которые искажают обучение;
- классы несбалансированы;
- отдельные признаки вообще не связаны с целевой переменной.

### Практический смысл EDA для ML

EDA помогает ответить на вопросы:

- какие признаки стоит оставить;
- какие признаки нужно преобразовать;
- как обрабатывать пропуски;
- какие алгоритмы будут уместны;
- какие метрики качества нужно использовать.

В качестве примера будем использовать датасет Breast Cancer Wisconsin из `scikit-learn`.  
Это задача **классификации**: по числовым признакам опухоли нужно определить класс.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

plt.rcParams["figure.figsize"] = (8, 5)
pd.set_option("display.max_columns", 100)

## 2. Загружаем данные и знакомимся с ними

### Зачем это делать

Первый шаг EDA — понять **структуру** данных:

- сколько объектов;
- сколько признаков;
- какие типы данных;
- что является целевой переменной;
- как выглядят первые строки.

Эта информация помогает быстро выявить явные ошибки и понять, как дальше строить анализ.

In [ ]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data.copy()
y = dataset.target.copy()

df = X.copy()
df["target"] = y
df["target_name"] = df["target"].map(dict(enumerate(dataset.target_names)))

print("Размер таблицы:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
print(df["target_name"].unique())

### Что мы уже узнали

- у нас **569 наблюдений**;
- почти все признаки — **числовые**;
- задача — **классификация**;
- целевая переменная хранится отдельно и может быть добавлена в таблицу для анализа;
- целевая переменная принимает два значения - 'malignant' (злокачественная) и 'benign' (доброкачественная), значит будем решать задачу бинарной классификации.

### Почему это важно для ML

После этого шага становится понятно:

- можно ли сразу обучать числовую модель;
- требуется ли кодирование категориальных признаков;
- достаточно ли наблюдений;
- есть ли технические проблемы в типах столбцов.

## 3. Пропуски и некорректные значения

### Зачем анализировать пропуски

Многие алгоритмы машинного обучения **не умеют работать с `NaN` напрямую**.  
Даже если модель технически обучается, пропуски могут искажать статистику и ухудшать качество.

### Зачем анализировать некорректные значения

Иногда в данных встречаются физически невозможные или подозрительные значения, например:

- отрицательный возраст;
- нулевая площадь квартиры;
- температура 5000 градусов;
- отрицательный размер опухоли.

Такие ошибки нужно обнаружить **до** обучения модели.

> Исходный датасет довольно чистый.  
> Для демонстрации методов ниже мы **специально внесём** несколько проблем в копию таблицы.

In [ ]:
rng = np.random.default_rng(42)
df_dirty = df.copy()

# Добавим искусственные пропуски
missing_idx_radius = rng.choice(df_dirty.index, size=12, replace=False)
missing_idx_texture = rng.choice(df_dirty.index, size=10, replace=False)

df_dirty.loc[missing_idx_radius, "mean radius"] = np.nan
df_dirty.loc[missing_idx_texture, "mean texture"] = np.nan

# Добавим несколько заведомо некорректных значений
bad_idx = rng.choice(df_dirty.index, size=3, replace=False)
df_dirty.loc[bad_idx, "mean area"] = -df_dirty.loc[bad_idx, "mean area"]

df_dirty.loc[sorted(bad_idx), ["mean area", "target_name"]]

In [ ]:
missing_counts = df_dirty.isna().sum().sort_values(ascending=False)
missing_counts[missing_counts > 0]

### Как интерпретировать результат

Мы видим, в каких столбцах есть пропуски и сколько их.

### Как это помогает при построении модели

После такого анализа можно принять решение:

- удалить строки с пропусками;
- заполнить пропуски средним, медианой, модой;
- использовать отдельный признак «значение отсутствовало»;
- отказаться от признака, если пропусков слишком много.

In [ ]:
# Поиск подозрительных значений: площадь не может быть отрицательной
df_dirty[df_dirty["mean area"] < 0][["mean radius", "mean area", "target_name"]]

### Что даёт этот шаг

Мы не просто находим «ошибки в таблице».  
Мы предотвращаем ситуацию, когда модель начинает учиться на артефактах данных, а не на реальных закономерностях.

## 4. Одномерный анализ признаков

### Зачем смотреть распределения

Для каждого признака полезно понять:

- в каком диапазоне он лежит;
- симметрично ли распределение;
- есть ли длинные хвосты;
- присутствуют ли экстремальные значения.

Это влияет на:

- выбор масштабирования;
- необходимость преобразований;
- устойчивость разных алгоритмов.

In [ ]:
desc = df.describe().T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
desc.head(10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df["mean radius"], bins=25)
axes[0].set_title("Распределение mean radius")
axes[0].set_xlabel("mean radius")
axes[0].set_ylabel("Частота")

axes[1].hist(df["mean texture"], bins=25)
axes[1].set_title("Распределение mean texture")
axes[1].set_xlabel("mean texture")
axes[1].set_ylabel("Частота")

axes[2].hist(df["mean area"], bins=25)
axes[2].set_title("Распределение mean area")
axes[2].set_xlabel("mean area")
axes[2].set_ylabel("Частота")

plt.tight_layout()
plt.show()

### Что можно понять по графикам

- некоторые признаки распределены относительно компактно;
- некоторые имеют заметную асимметрию;
- масштаб признаков сильно различается.

### Почему это важно для ML

Если признаки имеют очень разные масштабы, то для моделей вроде:

- логистической регрессии,
- метода k ближайших соседей,
- метода опорных векторов,

обычно требуется **масштабирование**.

Для деревьев решений масштабирование часто не так критично, но понимание распределений всё равно полезно.

## 5. Выбросы

### Зачем искать выбросы

Выбросы — это наблюдения, которые сильно отличаются от остальных.

Они могут быть:

- результатом ошибки измерения;
- редким, но реальным случаем;
- важным сигналом;
- источником нестабильности для модели.

Особенно чувствительны к выбросам линейные методы и методы, основанные на расстояниях.

Функция boxplot ниже строит «ящик с усами» (box-and-whisker plot) — график, компактно визуализирующий распределение числовых данных: медиану, квартили (25% и 75%), межквартильный размах, размах данных и выбросы. 

Основные элементы графика:
- Ящик (Box): показывает межквартильный размах $\text{IQR}=Q_3-Q_1$, охватывающий 50% центральных данных.
- Линия внутри ящика: обозначает медиану (50-й процентиль).
- Усы (Whiskers): линии, уходящие от ящика, показывают разброс данных в пределах $\pm 1.5 \times \text{IQR}$ от границ ящика.
- Выбросы (Outliers): отдельные точки за пределами «усов», обозначающие аномально маленькие или большие значения.

In [ ]:
plt.figure(figsize=(8, 4))
plt.boxplot(df["mean area"], vert=False)
plt.title("Boxplot для признака mean area")
plt.xlabel("mean area")
plt.show()

In [ ]:
q1 = df["mean area"].quantile(0.25)
q3 = df["mean area"].quantile(0.75)
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outliers = df[(df["mean area"] < lower) | (df["mean area"] > upper)]
print("Количество выбросов по правилу IQR:", len(outliers))
outliers[["mean area", "mean radius", "target_name"]].head()

### Как использовать это в ML

После обнаружения выбросов можно принять разные решения:

- оставить их как есть, если это реальные редкие случаи;
- удалить, если это явные ошибки;
- применить робастные методы;
- использовать преобразование признака;
- проверить качество модели с выбросами и без них.

Главный принцип: **не удалять выбросы автоматически**, а сначала понять их природу.

## 6. Анализ связей между признаками

### Зачем смотреть зависимости между признаками

Если два признака очень сильно связаны, они могут дублировать информацию.  
Это особенно важно для интерпретируемых линейных моделей, где мультиколлинеарность затрудняет анализ коэффициентов.

Также связь между признаками помогает лучше понять структуру данных.

In [ ]:
corr = df.drop(columns=["target", "target_name"]).corr(numeric_only=True)

# Покажем 10 самых сильных корреляций между разными признаками
corr_pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .sort_values(key=np.abs, ascending=False)
)

corr_pairs.head(10)

In [ ]:
top_feature_1, top_feature_2 = corr_pairs.index[0]

plt.figure(figsize=(7, 5))
plt.scatter(df[top_feature_1], df[top_feature_2], alpha=0.7)
plt.xlabel(top_feature_1)
plt.ylabel(top_feature_2)
plt.title(f"Связь между {top_feature_1} и {top_feature_2}")
plt.show()

### Что это даёт на практике

Если признаки почти дублируют друг друга, можно:

- оставить только один из них;
- применить методы снижения размерности;
- быть осторожнее при интерпретации коэффициентов линейной модели.

### Польза для модели

- упрощение модели;
- снижение риска переобучения;
- повышение интерпретируемости.

## 7. Связь признаков с целевой переменной

### Зачем анализировать признаки относительно target

EDA нужен не только для понимания самих признаков, но и для ответа на главный вопрос:

> **какие признаки действительно помогают предсказывать целевую переменную?**

Если распределения признака в разных классах заметно отличаются, это хороший сигнал: признак может быть полезен модели.

In [ ]:
df.groupby("target_name")[["mean radius", "mean texture", "mean area"]].mean()

In [ ]:
classes = df["target_name"].unique()

data_for_boxplot = [df.loc[df["target_name"] == cls, "mean radius"] for cls in classes]

plt.figure(figsize=(7, 5))
plt.boxplot(data_for_boxplot, tick_labels=classes)
plt.title("mean radius в разрезе классов")
plt.ylabel("mean radius")
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
for cls in classes:
    subset = df[df["target_name"] == cls]
    plt.scatter(subset["mean radius"], subset["mean texture"], alpha=0.6, label=cls)

plt.xlabel("mean radius")
plt.ylabel("mean texture")
plt.title("Признаки и целевая переменная")
plt.legend()
plt.show()

### Как это помогает в ML

На основе такого анализа можно:

- отобрать информативные признаки;
- создать новые признаки;
- упростить модель;
- понять, почему модель работает хорошо или плохо.

Если признак почти одинаково распределён во всех классах, его польза для классификации может быть ограниченной.

## 8. Дисбаланс классов

### Зачем это проверять

Если один класс встречается намного чаще другого, модель может показывать высокую **accuracy**, но при этом плохо находить редкий класс.

Сначала посмотрим на распределение классов в исходном датасете.

In [ ]:
class_counts = df["target_name"].value_counts()
class_counts

In [ ]:
plt.figure(figsize=(6, 4))
plt.bar(class_counts.index, class_counts.values)
plt.title("Распределение классов")
plt.ylabel("Количество объектов")
plt.show()

В исходном датасете дисбаланс не критичен, но он есть.  
Чтобы показать проблему нагляднее, создадим **искусственно несбалансированную** выборку.

In [ ]:
df_majority = df[df["target"] == 1].sample(500, random_state=42, replace=True)
df_minority = df[df["target"] == 0].sample(10, random_state=42, replace=True)

df_imbalanced = pd.concat([df_majority, df_minority], ignore_index=True)

X_imb = df_imbalanced[dataset.feature_names]
y_imb = df_imbalanced["target"]

print(y_imb.value_counts())

При работе с сильно несбалансированными данными accuracy может получиться очень высокой, но качество модели будет на самом деле низким.

В таких случаях используют дополнительные оценки: precision, recall, f1-score.

Существует два типа предсказаний: положительное и отрицательное. И каждое из них может быть истинным или ложным.

- Истинно положительный результат (true positive): модель предсказывает класс 1 и это действительно класс 1.
- Ложно положительный результат (false positive): модель предсказывает класс 1, но на самом деле это класс 0. Ошибка I рода, ложное обнаружение.
- Ложно отрицательный результат (false negative): модель предсказывает класс 0, но это неверно. Ошибка II рода, пропуск обнаружения.
- Истинно отрицательный результат (true negative): модель предсказывает класс 0 и это действительно так.

Все эти случаи можно собрать в таблицу, которая называется матрицей ошибок.

| | Класс 1 | Класс 0 |
|-|-|-|
| Предсказан класс 1 | TP (число <br>истинно положительных случаев)  | FP (число<br>ложноположительных случаев) |
| Предсказан класс 0 | FN (число <br>ложноотрицательных случаев) | TN (число <br>истинно отрицательных случаев) |

Precision — это доля истинно положительных предсказаний в общем числе положительных предсказаний.

$$
	\text{Precision} = \frac{	\text{TP}}{	\text{TP} + 	\text{FP}}
$$

Recall — это доля истинно положительных предсказаний в общем числе положительных случаев.

$$
	\text{Recall} = \frac{	\text{TP}}{	\text{TP} + 	\text{FN}}
$$

По этим двум значениям вычисляют их гармоническое среднее, которое называется F1-score:

$$
	\text{F1-score} = 2\frac{	\text{Precision} 	\times 	\text{Recall}}{	\text{Precision} + 	\text{Recall}}
$$

Ещё более детальный анализ можно выполнить, если вычислить precision и recall по каждому классу отдельно (число истинных предсказаний класса 1, число истинных предсказаний класса 0 и так далее). 

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_imb, y_imb, test_size=0.25, stratify=y_imb, random_state=42
)

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, digits=4))

В приведённом примере модель показывает очень высокую accuaracy=0.98, но более тонкий анализ показывает, что на самом деле её качество низкое. Об этом говорит низкое значение recall.

### Главный вывод

Даже если `accuracy` выглядит высокой, важно смотреть на:

- precision,
- recall,
- f1-score,
- confusion matrix.

### Польза для ML

Зная о наличии дисбаланса, нужно быть готовым что `accuracy`, скорее всего, будет давать неадекватную оценку. Тогда при работе с модельно нужно будет:

- использовать другие метрики;
- применять `class_weight`;
- использовать oversampling / undersampling (удаление данных за счёт сглаживания или добавление за счёт интерполяции);
- отдельно контролировать качество на редком классе.

## 9. Небольшой мини-кейс: как результаты EDA влияют на модель

Теперь покажем простую связь между EDA и обучением модели.

Сравним три ситуации:

1. попытка обучить модель на исходных «грязных» данных без какой-либо обработки;
2. наивный подход: удалить строки с пропусками, но не исправлять подозрительные значения;
3. осмысленная базовая очистка после EDA: исправить ошибки, заполнить пропуски и масштабировать признаки.


In [ ]:
# Подготовим таблицу для эксперимента
feature_cols = list(dataset.feature_names)

X_dirty = df_dirty[feature_cols].copy()
y_dirty = df_dirty["target"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X_dirty, y_dirty, test_size=0.25, stratify=y_dirty, random_state=42
)

print("=== 1. Попытка обучить модель на неочищенных данных ===")
raw_model = LogisticRegression(max_iter=1000)

try:
    raw_model.fit(X_train, y_train)
    y_pred_raw = raw_model.predict(X_test)
    print("Accuracy на неочищенных данных:", round(accuracy_score(y_test, y_pred_raw), 4))
except Exception as e:
    print("Модель не обучилась на неочищенных данных.")
    print("Причина:", e)

print("\n=== 2. Наивный вариант без полноценного EDA ===")
# Удаляем строки с пропусками, но оставляем подозрительные отрицательные значения
train_naive = pd.concat([X_train, y_train], axis=1).dropna()
test_naive = pd.concat([X_test, y_test], axis=1).dropna()

X_train_naive = train_naive[feature_cols]
y_train_naive = train_naive["target"]
X_test_naive = test_naive[feature_cols]
y_test_naive = test_naive["target"]

# Модель обучается, но выдаётся предупреждение о плохой сходимости 
# Взято max_iter=1111 чтобы сразу было видно, что предупреждения появляется здесь
naive_model = LogisticRegression(max_iter=1111) 
naive_model.fit(X_train_naive, y_train_naive)
y_pred_naive = naive_model.predict(X_test_naive)

print("Размер train после простого удаления пропусков:", X_train_naive.shape)
print("Размер test после простого удаления пропусков:", X_test_naive.shape)
print("Accuracy при наивной обработке:", round(accuracy_score(y_test_naive, y_pred_naive), 4))

print("\n=== 3. Осмысленная обработка после EDA ===")
X_train_clean = X_train.copy()
X_test_clean = X_test.copy()

# Исправим физически невозможные отрицательные значения:
# заменим их на NaN, а затем восстановим медианой
X_train_clean.loc[X_train_clean["mean area"] < 0, "mean area"] = np.nan
X_test_clean.loc[X_test_clean["mean area"] < 0, "mean area"] = np.nan

clean_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

clean_pipeline.fit(X_train_clean, y_train)
y_pred_clean = clean_pipeline.predict(X_test_clean)

print("Accuracy после базовой обработки:", round(accuracy_score(y_test, y_pred_clean), 4))

### Что показывает этот пример

- **Без очистки данных модель может вообще не обучиться**, потому что многие алгоритмы не умеют работать с пропусками.
- **Наивная обработка** (например, просто удалить строки с пропусками) позволяет получить результат, но часть данных теряется, а ошибки в признаках остаются.
- **Осмысленная обработка после EDA** использует выводы анализа данных: мы замечаем пропуски, находим некорректные значения, выбираем способ их исправления и только после этого обучаем модель.

Именно поэтому EDA — это практический этап подготовки модели, а не просто визуальное описание данных.


## 10. Какие практические решения можно принять после EDA

Ниже приведена типичная связь между наблюдениями на этапе EDA и действиями на этапе ML.

| Наблюдение в EDA | Практическое решение |
|---|---|
| Есть пропуски | Заполнение, удаление, отдельный индикатор пропуска |
| Есть некорректные значения | Исправление, удаление, замена на NaN |
| Есть выбросы | Проверка природы выбросов, робастные методы, фильтрация |
| Сильная асимметрия признака | Логарифмирование, нормализация |
| Разные масштабы признаков | Масштабирование |
| Высокая корреляция признаков | Отбор признаков, снижение размерности |
| Признак слабо связан с target | Рассмотреть исключение признака |
| Сильный дисбаланс классов | Другие метрики, балансировка, веса классов |

## 11. Краткие выводы

Разведочный анализ данных нужен для того, чтобы **понять данные до построения модели**.

### Главное, что стоит запомнить

1. EDA помогает обнаружить проблемы в данных до обучения модели.  
2. Каждый шаг EDA имеет прикладной смысл для ML.  
3. Цель EDA — не просто «посмотреть на графики», а принять решения:
   - как очищать данные;
   - какие признаки использовать;
   - какие преобразования применять;
   - какую модель и какие метрики выбирать.

### Итоговая идея

**Хорошая модель начинается не с выбора алгоритма, а с понимания данных.**

## 12. Контрольные вопросы

1. Почему пропуски нельзя игнорировать?  
2. Всегда ли выбросы нужно удалять?  
3. Почему высокая accuracy может быть обманчивой при дисбалансе классов?  
4. Зачем анализировать корреляции между признаками?  
5. Почему EDA проводится до обучения модели, а не после?